# Visão Probabilística, Cross-Validation e Bootstrapping

**Statistisches Lernen 2 — FH Kufstein Tirol**  
Referência: `lectures/Statistisches_Lernen_2_v4.pdf`

Este notebook cobre a **perspectiva probabilística** do machine learning, que revela por que MSE e regularização emergem naturalmente do Teorema de Bayes. Depois, explora **Cross-Validation** e **Bootstrapping** como ferramentas práticas de avaliação e seleção de modelos.

---

> **Estrutura de cada seção:** Conceito Teórico → Matemática (LaTeX) → Why & How → Hands-on (Python)

---

## Conteúdo

| # | Seção | Tópicos |
|---|-------|---------|
| 1 | Probabilistic View | Likelihood, Posterior, MAP |
| 2 | Prior Choices | Gaussian Prior → L2, Laplace Prior → L1 |
| 3 | Decision Process | Tabela de cenários de regularização |
| 4 | Hyperparameter Choice | Log-grid, Coarse-to-Fine, Protocol |
| 5 | Diagnosis | Under vs. Over-Regularization |
| 6 | Cross-Validation | Hold-Out, K-Fold, Stratified, LOO |
| 7 | Bootstrapping | Resampling, Confidence Intervals, Bagging |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import (KFold, StratifiedKFold, LeaveOneOut,
                                      cross_val_score, learning_curve,
                                      train_test_split)
from sklearn.datasets import make_classification, make_regression
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.pipeline import Pipeline

np.random.seed(42)
plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["font.size"] = 11

---

## Seção 1 — Probabilistic View (Visão Probabilística)

### 1.1 Conceito Teórico

Até agora, o treinamento de um modelo era visto como **minimizar um erro** (ex: MSE). A **Probabilistic View** pergunta algo diferente:

> *Dado que observamos os dados $D = \{(x_i, y_i)\}$, qual conjunto de parâmetros $\theta$ é mais **provável**?*

Essa mudança de perspectiva conecta dois conceitos fundamentais:

| Conceito | Pergunta que responde |
|----------|----------------------|
| **Likelihood** $p(D \mid \theta)$ | "Se o modelo tivesse estes parâmetros, quão prováveis seriam os dados?" |
| **Posterior** $p(\theta \mid D)$ | "Dado que vi estes dados, quão provável é cada configuração de $\theta$?" |

O objetivo é maximizar o **Posterior** — isto é chamado de estimativa **MAP (Maximum A Posteriori)**.

### 1.2 Matemática

**Passo 1 — Likelihood (Verossimilhança)**

Para um dataset $D = \{(x_i, y_i)\}_{i=1}^N$ e um modelo $f_\theta$, assumimos ruído Gaussiano independente por amostra:

$$p(D \mid \theta) = \prod_{i=1}^{N} p((x_i, y_i) \mid \theta) = \prod_{i=1}^{N} \exp\!\left(-\|f_\theta(x_i) - y_i\|^2\right)$$

*(As variâncias $\sigma^2$ são omitidas — as igualdades valem a menos de uma constante multiplicativa)*

Tirando o logaritmo negativo (equivale a minimizar, pois log é monotônico):

$$-\log p(D \mid \theta) = \sum_{i=1}^{N} \|f_\theta(x_i) - y_i\|^2 \quad \Longleftrightarrow \quad \text{MSE / SSE}$$

**Passo 2 — Teorema de Bayes**

$$\boxed{p(\theta \mid D) \propto p(D \mid \theta) \cdot p(\theta)}$$

| Termo | Nome |
|-------|------|
| $p(D \mid \theta)$ | Likelihood |
| $p(\theta)$ | Model Prior |
| $p(\theta \mid D)$ | Posterior |

**Passo 3 — Minimizar o negativo do log do Posterior**

Maximizar $p(\theta \mid D)$ é equivalente a minimizar $-\log p(\theta \mid D)$:

$$-\log p(\theta \mid D) = \underbrace{-\log p(D \mid \theta)}_{\text{erro nos dados (MSE)}} + \underbrace{-\log p(\theta)}_{\text{regularização}}$$

Esta é exatamente a forma que usamos no treinamento com **loss + regularizador**!

### 1.3 Why & How

**Por que isso importa?**
- O **MSE não é arbitrário** — ele emerge diretamente da suposição de ruído Gaussiano.
- A **regularização não é um truque** — ela é a incorporação de conhecimento prévio ($p(\theta)$) sobre os parâmetros.
- A escolha do **prior** determina o tipo de regularização:
  - Prior Gaussiano → Regularização L2 (Ridge)
  - Prior de Laplace → Regularização L1 (Lasso)

**Como usar na prática:**
1. Defina seu modelo $f_\theta$ (linear, neural, etc.)
2. Escolha a distribuição de ruído (Gaussiana → MSE, Bernoulli → cross-entropy)
3. Escolha um prior sobre $\theta$ (Gaussiano, Laplace, ou nenhum = MLE)
4. Minimize $-\log p(D|\theta) - \log p(\theta)$ por otimização numérica

In [ ]:
# === Hands-on: Simular dados com outlier e comparar MLE vs MAP ===

theta_real = np.array([0.5, 1.2])  # intercepto e inclinação reais

x = np.array([-1.5, -0.8, -0.2, 0.3, 0.9, 1.5])
ruido = np.array([0.05, -0.20, 0.15, -0.05, 0.10, 1.10])  # último é outlier
y = theta_real[0] + theta_real[1] * x + ruido

X_mat = np.column_stack([np.ones_like(x), x])   # Design Matrix [1, x]

# MLE = mínimos quadrados (sem prior)
theta_mle = np.linalg.solve(X_mat.T @ X_mat, X_mat.T @ y)

def neg_log_likelihood(theta, X, y, sigma=0.8):
    residuos = y - X @ theta
    return np.sum(residuos**2) / (2 * sigma**2)

def gradient_map(theta, X, y, lam, tipo="l2", sigma=0.8):
    residuos = y - X @ theta
    grad = -(X.T @ residuos) / sigma**2
    if tipo == "l2":
        grad += lam * theta
    elif tipo == "l1":
        grad += lam * np.sign(theta)
    return grad

def ajustar_map(X, y, lam, tipo="l2", n_passos=8000, taxa=0.01):
    theta = np.zeros(X.shape[1])
    for _ in range(n_passos):
        theta -= taxa * gradient_map(theta, X, y, lam, tipo)
    return theta

lam = 2.0
theta_map_l2 = ajustar_map(X_mat, y, lam=lam, tipo="l2")
theta_map_l1 = ajustar_map(X_mat, y, lam=lam, tipo="l1")

print("Parâmetros estimados:")
print(f"  Real (simulado): {theta_real}")
print(f"  MLE  (sem prior): {theta_mle.round(3)}")
print(f"  MAP L2 (λ={lam}): {theta_map_l2.round(3)}")
print(f"  MAP L1 (λ={lam}): {theta_map_l1.round(3)}")

x_plot = np.linspace(-1.8, 1.8, 200)
X_plot = np.column_stack([np.ones(200), x_plot])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.scatter(x, y, s=80, color="tab:blue", zorder=4, label="dados (treino)")
ax.scatter(x[-1], y[-1], s=140, color="red", zorder=5, marker="*", label="outlier")
ax.plot(x_plot, X_plot @ theta_real, "k--", alpha=0.4, label="real (simulado)")
ax.plot(x_plot, X_plot @ theta_mle, "r-", lw=2, label="MLE (sem prior)")
ax.plot(x_plot, X_plot @ theta_map_l2, "g-", lw=2, label=f"MAP L2 (λ={lam})")
ax.plot(x_plot, X_plot @ theta_map_l1, "m-", lw=2, label=f"MAP L1 (λ={lam})")
ax.set_title("Efeito do prior: MLE vs MAP")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

ax = axes[1]
nomes = ["MLE\n(sem prior)", f"MAP L2\n(λ={lam})", f"MAP L1\n(λ={lam})", "Real"]
incls = [theta_mle[1], theta_map_l2[1], theta_map_l1[1], theta_real[1]]
cores = ["red", "green", "purple", "black"]
bars = ax.bar(nomes, incls, color=cores, alpha=0.7)
ax.axhline(theta_real[1], color="black", ls="--", lw=1)
ax.set_title("Inclinação estimada θ₁")
ax.set_ylabel("θ₁")
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print("\nInsight: o prior 'puxa' os parâmetros para zero, reduzindo o efeito do outlier.")

---

## Seção 2 — Prior Choices: Gaussian Prior → L2, Laplace Prior → L1

### 2.1 Conceito Teórico

A escolha do **Prior** $p(\theta)$ determina qual tipo de regularização surge na otimização. O slide apresenta dois priors padrão:

| Prior | Distribuição | Regularização resultante |
|-------|-------------|-------------------------|
| **Gaussiano** | $\mathcal{N}(0, \tau^2 I)$ | **L2 / Ridge / Weight Decay** |
| **Laplace** | $\text{Laplace}(0, b)$ | **L1 / Lasso / Basis Pursuit** |

### 2.2 Matemática

**Prior Gaussiano (→ L2):**

$$p(\theta) = \exp\!\left(-\|\theta\|^2\right) \quad \Rightarrow \quad -\log p(\theta) = \|\theta\|^2 = \sum_j \theta_j^2$$

Logo, o objetivo MAP com prior Gaussiano é:

$$\min_\theta \; \sum_{i=1}^N \|f_\theta(x_i) - y_i\|^2 + \lambda \|\theta\|^2 \quad \text{(Ridge)}$$

onde $\lambda = \sigma^2 / \tau^2$ conecta a força do prior com a força da regularização.

**Prior de Laplace (→ L1):**

$$p(\theta) = \exp\!\left(-\|\theta\|_1\right) \quad \Rightarrow \quad -\log p(\theta) = \|\theta\|_1 = \sum_j |\theta_j|$$

Logo, o objetivo MAP com prior de Laplace é:

$$\min_\theta \; \sum_{i=1}^N \|f_\theta(x_i) - y_i\|^2 + \lambda \|\theta\|_1 \quad \text{(Lasso)}$$

**Diferença geométrica:** a Gaussiana tem caudas leves (penaliza parâmetros grandes suavemente); a Laplace tem caudas pesadas e uma cúspide em zero (induz **sparsity** — força parâmetros exatamente para zero).

### 2.3 Why & How

- Use **L2 (Prior Gaussiano)** quando: você acredita que todos os parâmetros são relevantes mas devem ser pequenos. Estável numericamente, solução analítica $(X^TX + \lambda I)^{-1}X^Ty$.
- Use **L1 (Prior de Laplace)** quando: você acredita que muitos parâmetros são irrelevantes (deseja **feature selection** automática). Produz soluções esparsas — muitos $\theta_j = 0$ exatamente.
- Ambos os priors são centrados em zero: expressam a crença de que parâmetros menores são mais prováveis a priori.

In [ ]:
# === Hands-on: Visualizar Prior Gaussiano vs Prior de Laplace ===

theta_vals = np.linspace(-4, 4, 400)

prior_gaussiano = stats.norm(loc=0, scale=1).pdf(theta_vals)
prior_laplace   = stats.laplace(loc=0, scale=1).pdf(theta_vals)
neg_log_gauss   = -np.log(prior_gaussiano + 1e-12)
neg_log_laplace = -np.log(prior_laplace   + 1e-12)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(theta_vals, prior_gaussiano, "g-",  lw=2.5, label="Gaussiano → L2")
ax.plot(theta_vals, prior_laplace,   "m--", lw=2.5, label="Laplace → L1")
ax.fill_between(theta_vals, prior_laplace, alpha=0.08, color="purple")
ax.fill_between(theta_vals, prior_gaussiano, alpha=0.08, color="green")
ax.set_title("Prior $p(\\theta)$")
ax.set_xlabel("θ")
ax.set_ylabel("probabilidade")
ax.annotate("Laplace tem cúspide\nem zero → sparsity",
            xy=(0, prior_laplace[200]), xytext=(1.0, 0.45),
            arrowprops=dict(arrowstyle="->", color="purple"),
            color="purple", fontsize=9)
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(theta_vals, neg_log_gauss,   "g-",  lw=2.5, label="-log prior Gaussiano = $\\|\\theta\\|^2$ (L2)")
ax.plot(theta_vals, neg_log_laplace, "m--", lw=2.5, label="-log prior Laplace = $\\|\\theta\\|_1$ (L1)")
ax.set_ylim(0, 10)
ax.set_title("-log Prior = Penalidade de Regularização")
ax.set_xlabel("θ")
ax.set_ylabel("-log p(θ) = penalidade")
ax.legend(); ax.grid(alpha=0.3)

plt.suptitle("Prior → Regularização: conexão Bayesiana", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Demonstrar sparsity do L1
X_sp, y_sp = make_regression(n_samples=80, n_features=20, n_informative=5,
                              noise=10, random_state=0)
ridge = Ridge(alpha=1.0).fit(X_sp, y_sp)
lasso = Lasso(alpha=1.0, max_iter=5000).fit(X_sp, y_sp)

fig, ax = plt.subplots(figsize=(11, 4))
idx = np.arange(20)
ax.bar(idx - 0.2, ridge.coef_, 0.4, label=f"Ridge  (zeros: {np.sum(np.abs(ridge.coef_)<0.01)})", color="green", alpha=0.7)
ax.bar(idx + 0.2, lasso.coef_, 0.4, label=f"Lasso  (zeros: {np.sum(lasso.coef_==0)})", color="purple", alpha=0.7)
ax.axhline(0, color="k", lw=0.8)
ax.set_title("Sparsity: Lasso força coeficientes exatamente a zero (Feature Selection automática)")
ax.set_xlabel("Feature index")
ax.set_ylabel("Coeficiente")
ax.legend(); ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

---

## Seção 3 — Decision Process: Quando usar qual Regularização?

### 3.1 Conceito Teórico

Não existe um método universal. O slide apresenta um **processo de decisão** baseado nas características do problema:

| Szenario | Empfehlung (Recomendação) | Hinweise (Dicas) |
|----------|--------------------------|------------------|
| **Small data — many features** | L2/L1, Early Stopping, arquitetura simples | Standardization + Cross-Validation para λ |
| **Structured Outputs** (imagens, séries) | TV / Laplace (Output Regularization) | Output Regularization é frequentemente útil |
| **Strong Prior Knowledge** | Regularization by Design | Limitar capacidade, usar inductive bias |
| **Optimization unstable** | Gradient Clipping, Max-Norm, Non-Negativity | Usar Learning Rate menor |

### 3.2 Why & How

- **Small data / many features:** o risco de overfitting é alto. L2 ou L1 com Cross-Validation para escolher λ é o ponto de partida seguro.
- **Structured Outputs:** saídas como pixels de imagem têm estrutura espacial. A regularização de vizinhança (TV — Total Variation) preserva bordas enquanto suaviza ruído.
- **Strong Prior Knowledge:** se você sabe que a solução tem uma forma específica (ex: sinal não-negativo, parâmetros simétricos), incorpore isso como constraint duro ou regularizador customizado.
- **Optimization unstable:** gradientes explodem? Use gradient clipping. Pesos crescem sem limite? Max-Norm limita a norma máxima por camada.

In [ ]:
# === Hands-on: Demonstrar cenário Small Data / Many Features ===

np.random.seed(7)
N_small, p_features = 60, 30   # poucos dados, muitas features

X_small, y_small = make_regression(n_samples=N_small, n_features=p_features,
                                    n_informative=8, noise=15, random_state=7)
scaler = StandardScaler()
X_small = scaler.fit_transform(X_small)

lambdas = np.logspace(-4, 2, 50)
cv = KFold(n_splits=5, shuffle=True, random_state=0)

cv_ridge = [-cross_val_score(Ridge(alpha=l), X_small, y_small,
                              cv=cv, scoring="neg_mean_squared_error").mean()
            for l in lambdas]
cv_lasso = [-cross_val_score(Lasso(alpha=l, max_iter=5000), X_small, y_small,
                              cv=cv, scoring="neg_mean_squared_error").mean()
            for l in lambdas]

best_l_ridge = lambdas[np.argmin(cv_ridge)]
best_l_lasso = lambdas[np.argmin(cv_lasso)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.semilogx(lambdas, cv_ridge, "g-", lw=2, label="Ridge (L2)")
ax.semilogx(lambdas, cv_lasso, "m--", lw=2, label="Lasso (L1)")
ax.axvline(best_l_ridge, color="green", ls=":", alpha=0.7, label=f"λ* Ridge={best_l_ridge:.1e}")
ax.axvline(best_l_lasso, color="purple", ls=":", alpha=0.7, label=f"λ* Lasso={best_l_lasso:.1e}")
ax.set_title(f"Small Data (N={N_small}) / Many Features (p={p_features})\nCV-MSE por λ")
ax.set_xlabel("λ (escala log)"); ax.set_ylabel("CV-MSE")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

ax = axes[1]
ridge_best = Ridge(alpha=best_l_ridge).fit(X_small, y_small)
lasso_best = Lasso(alpha=best_l_lasso, max_iter=5000).fit(X_small, y_small)
idx = np.arange(p_features)
ax.bar(idx - 0.2, ridge_best.coef_, 0.4, color="green", alpha=0.7, label="Ridge (todos pequenos)")
ax.bar(idx + 0.2, lasso_best.coef_, 0.4, color="purple", alpha=0.7,
       label=f"Lasso ({np.sum(lasso_best.coef_!=0)} features ativas)")
ax.axhline(0, color="k", lw=0.5)
ax.set_title("Coeficientes com λ* ótimo")
ax.set_xlabel("Feature index"); ax.set_ylabel("Coeficiente")
ax.legend(fontsize=9); ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print(f"λ* Ridge: {best_l_ridge:.2e}  |  λ* Lasso: {best_l_lasso:.2e}")
print(f"Lasso manteve apenas {np.sum(lasso_best.coef_!=0)} de {p_features} features.")

---

## Seção 4 — Hyperparameter Choice (Escolha de Hiperparâmetros)

### 4.1 Conceito Teórico

Após escolher o tipo de regularização, é preciso **encontrar o valor ótimo de λ** sem contaminar o conjunto de teste. O slide apresenta um processo estruturado em três eixos:

**Normalização:**
- Dividir o regularizador de saída pelo número de componentes (ex: pixels) → λ fica invariante à escala
- Para somas sobre amostras: usar **média** em vez de soma

**Busca:**
- Grid **logarítmico** (não linear) — λ age multiplicativamente, não aditivamente
- Estratégia **Coarse-to-Fine**: busca ampla → refina na região prometedora
- Custo-efetivo: poucas épocas para a busca ampla, mais épocas apenas no refinamento

**Estabilidade:**
- Optimizadores estáveis (Adam, SGD com momentum)
- Early Stopping como regularizador adicional

### 4.2 Hyperparameter Guidelines (valores do slide)

| Hiperparâmetro | Faixa típica |
|----------------|--------------|
| **L²** (Ridge / Weight Decay) | `1e-6 … 1e-2` |
| **L¹** (Lasso) | `1e-6 … 1e-3` |
| **Max-Norm** (por camada) | `1 … 3` |
| **Dropout-rate** | `0.1 … 0.5` |
| **Patience** (Early Stopping) | `5 … 10` épocas |
| **Learning Rate Schedule** | ajudar sempre |

### 4.3 Hyperparameter Choice Protocol (slide)

1. **Split limpo:** Train / Val / Test — sem contaminação; estratificação para classificação
2. **Coarse sweep:** cada regularizador com λ em log-grid amplo
3. **Refinamento:** apenas nas top 2–3 regiões promissoras
4. **Média sobre seeds:** 3–5 seeds se N for pequeno
5. **Modelo final:** retreinar em Train+Val com o melhor λ
6. **Teste:** reportar no conjunto de teste **apenas uma vez**

In [ ]:
# === Hands-on: Coarse-to-Fine Log-Grid Hyperparameter Search ===

np.random.seed(0)
X_hp, y_hp = make_regression(n_samples=200, n_features=15, n_informative=6,
                              noise=20, random_state=0)
X_hp = StandardScaler().fit_transform(X_hp)

X_tr, X_val, y_tr, y_val = train_test_split(X_hp, y_hp, test_size=0.25, random_state=0)

# ---- Fase 1: Coarse sweep ----
lambdas_coarse = np.logspace(-6, 2, 30)
mse_coarse = []
for lam in lambdas_coarse:
    m = Ridge(alpha=lam).fit(X_tr, y_tr)
    mse_coarse.append(np.mean((y_val - m.predict(X_val))**2))

idx_best_coarse = np.argmin(mse_coarse)
melhor_coarse = lambdas_coarse[idx_best_coarse]

# ---- Fase 2: Fine sweep ao redor do melhor λ coarse ----
lambdas_fine = np.logspace(np.log10(melhor_coarse) - 1,
                            np.log10(melhor_coarse) + 1, 40)
mse_fine = []
for lam in lambdas_fine:
    m = Ridge(alpha=lam).fit(X_tr, y_tr)
    mse_fine.append(np.mean((y_val - m.predict(X_val))**2))

melhor_fine = lambdas_fine[np.argmin(mse_fine)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.semilogx(lambdas_coarse, mse_coarse, "o-", color="steelblue", lw=2, ms=5)
ax.axvline(melhor_coarse, color="red", ls="--", label=f"λ* coarse = {melhor_coarse:.2e}")
ax.axvspan(melhor_coarse/10, melhor_coarse*10, alpha=0.1, color="orange", label="região para refinamento")
ax.set_title("Fase 1: Coarse Sweep (log-grid amplo)")
ax.set_xlabel("λ"); ax.set_ylabel("Val-MSE")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

ax = axes[1]
ax.semilogx(lambdas_fine, mse_fine, "s-", color="darkorange", lw=2, ms=5)
ax.axvline(melhor_fine, color="red", ls="--", label=f"λ* fine = {melhor_fine:.2e}")
ax.set_title("Fase 2: Fine Sweep (refinamento local)")
ax.set_xlabel("λ"); ax.set_ylabel("Val-MSE")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.suptitle("Coarse-to-Fine Log-Grid Search — Ridge (L2)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"λ* Coarse: {melhor_coarse:.3e}")
print(f"λ* Fine:   {melhor_fine:.3e}")

In [ ]:
# === Multi-seed averaging para N pequeno ===

N_tiny = 40
X_t, y_t = make_regression(n_samples=N_tiny, n_features=10, n_informative=4, noise=15, random_state=5)
X_t = StandardScaler().fit_transform(X_t)

lambdas_test = np.logspace(-5, 2, 40)
n_seeds = 5
all_mse = np.zeros((n_seeds, len(lambdas_test)))

for s in range(n_seeds):
    Xtr, Xv, ytr, yv = train_test_split(X_t, y_t, test_size=0.25, random_state=s)
    for j, lam in enumerate(lambdas_test):
        m = Ridge(alpha=lam).fit(Xtr, ytr)
        all_mse[s, j] = np.mean((yv - m.predict(Xv))**2)

media_mse = all_mse.mean(axis=0)
std_mse   = all_mse.std(axis=0)

fig, ax = plt.subplots(figsize=(10, 5))
for s in range(n_seeds):
    ax.semilogx(lambdas_test, all_mse[s], alpha=0.25, color="gray")
ax.semilogx(lambdas_test, media_mse, "b-", lw=2.5, label="Média (5 seeds)")
ax.fill_between(lambdas_test, media_mse - std_mse, media_mse + std_mse,
                alpha=0.2, color="blue", label="± 1 std")
ax.axvline(lambdas_test[np.argmin(media_mse)], color="red", ls="--",
           label=f"λ* médio = {lambdas_test[np.argmin(media_mse)]:.2e}")
ax.set_title(f"Multi-seed averaging (N={N_tiny}) — reduz variabilidade da escolha de λ")
ax.set_xlabel("λ"); ax.set_ylabel("Val-MSE")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---

## Seção 5 — Diagnosis: Under vs. Over-Regularization

### 5.1 Conceito Teórico

Após escolher λ, é essencial **diagnosticar** se a regularização está corretamente calibrada. O slide apresenta três cenários:

**Under-Regularization (λ muito pequeno):**
- Alta variância: grande gap entre Train e Val
- Fronteiras de decisão instáveis
- **Ação:** aumentar λ; reduzir capacidade; Early Stopping mais agressivo

**Over-Regularization (λ muito alto):**
- Alto bias: Train e Val ambos ruins
- Saídas muito suaves / borradas
- **Ação:** reduzir λ; aumentar capacidade; Data Augmentation pode ajudar

**Output Regularization (TV — Total Variation):**
- TV demais: estruturas super-suavizadas, sem detalhes finos
- TV de menos: ruído de alta frequência nas saídas

### 5.2 Matemática — Learning Curve

A **Learning Curve** plota o erro de treino e validação em função do tamanho do dataset $N$:

$$\text{gap}(N) = \text{Val-MSE}(N) - \text{Train-MSE}(N)$$

- **Under-regularização:** gap grande para todo N → model high variance
- **Over-regularização:** gap pequeno mas ambos altos → model high bias
- **Bom λ:** gap fecha à medida que N cresce; erros convergem para nível aceitável

In [ ]:
# === Hands-on: Learning Curves para Under / Correto / Over-regularization ===

np.random.seed(1)
X_diag, y_diag = make_regression(n_samples=300, n_features=20, n_informative=8,
                                   noise=25, random_state=1)
X_diag = StandardScaler().fit_transform(X_diag)

cenarios = [
    ("Under-Regularização\n(λ=1e-6, alto variance)", Ridge(alpha=1e-6), "red"),
    ("Regularização Correta\n(λ=1.0)", Ridge(alpha=1.0), "green"),
    ("Over-Regularização\n(λ=1e4, alto bias)", Ridge(alpha=1e4), "blue"),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
tamanhos = np.linspace(0.1, 1.0, 10)

for ax, (titulo, modelo, cor) in zip(axes, cenarios):
    cv_k = KFold(n_splits=5, shuffle=True, random_state=2)
    sizes, tr_scores, val_scores = learning_curve(
        modelo, X_diag, y_diag,
        train_sizes=tamanhos,
        cv=cv_k,
        scoring="neg_mean_squared_error",
        n_jobs=-1
    )
    tr_mse  = -tr_scores.mean(axis=1)
    val_mse = -val_scores.mean(axis=1)

    ax.plot(sizes, tr_mse,  "o-", color=cor,   lw=2, label="Train MSE")
    ax.plot(sizes, val_mse, "s--", color="gray", lw=2, label="Val MSE")
    ax.fill_between(sizes, tr_mse, val_mse, alpha=0.15, color=cor)
    ax.set_title(titulo, fontsize=10)
    ax.set_xlabel("N amostras de treino")
    ax.set_ylabel("MSE")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    gap = val_mse[-1] - tr_mse[-1]
    ax.annotate(f"gap = {gap:.0f}", xy=(sizes[-1], (val_mse[-1]+tr_mse[-1])/2),
                ha="right", fontsize=9, color=cor)

plt.suptitle("Diagnosis via Learning Curves", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("Interpretação:")
print("  Under-regularização: gap Train/Val GRANDE → overfitting → aumentar λ")
print("  Correta:             gap pequeno, ambos razoáveis → bom equilíbrio")
print("  Over-regularização:  gap pequeno MAS ambos ALTOS → underfitting → reduzir λ")

---

## Seção 6 — Cross-Validation

### 6.1 Conceito Teórico

**Cross-Validation** é uma família de técnicas para estimar como um modelo de machine learning **performa em dados não vistos** durante o treinamento.

Pode ser usada para:
- **Avaliar** um modelo (estimar performance real)
- **Selecionar** o melhor modelo entre candidatos
- **Ajustar hiperparâmetros** (Parameter Tuning)
- **Evitar overfitting** ao conjunto de validação

**Como funciona:**
1. O dataset é dividido em vários subconjuntos
2. O modelo treina em alguns deles e é testado nos restantes
3. Esse processo se repete com diferentes combinações de subconjuntos
4. A performance final é a **média** de todas as rodadas de validação

### 6.2 Variantes

| Variante | Descrição | Quando usar |
|----------|-----------|-------------|
| **Hold-Out** | 70–80% treino / 20–30% validação | N grande (>10k); rápido |
| **K-Fold** | K folds; treina em K−1, testa em 1; repete K vezes | Padrão geral (K=5 ou 10) |
| **Stratified K-Fold** | Como K-Fold mas preserva proporção de classes | Classificação com classes desbalanceadas |
| **Leave-One-Out (LOO)** | Cada ponto é test set uma vez; N treinos | N muito pequeno; custo elevado |

### 6.3 Matemática

Para K-Fold, a estimativa de erro é:

$$\hat{\text{Err}}_{K\text{-Fold}} = \frac{1}{K} \sum_{k=1}^{K} \mathcal{L}\!\left(f_{-k}, D_k\right)$$

onde $f_{-k}$ é o modelo treinado em todos os folds exceto o $k$-ésimo, e $\mathcal{L}$ é a loss no fold $k$.

In [ ]:
# === Hands-on: Comparar as 4 variantes de Cross-Validation ===

np.random.seed(3)
X_cv, y_cv = make_regression(n_samples=120, n_features=10, n_informative=5,
                               noise=20, random_state=3)
X_cv = StandardScaler().fit_transform(X_cv)

modelo_cv = Ridge(alpha=1.0)

# Hold-Out (30 repetições para estimar variabilidade)
holdout_scores = []
for seed in range(30):
    Xtr, Xvl, ytr, yvl = train_test_split(X_cv, y_cv, test_size=0.25, random_state=seed)
    modelo_cv.fit(Xtr, ytr)
    holdout_scores.append(-np.mean((yvl - modelo_cv.predict(Xvl))**2))

# K-Fold (K=5)
kf5  = KFold(n_splits=5,  shuffle=True, random_state=0)
kf10 = KFold(n_splits=10, shuffle=True, random_state=0)

scores_kf5  = cross_val_score(modelo_cv, X_cv, y_cv, cv=kf5,  scoring="neg_mean_squared_error")
scores_kf10 = cross_val_score(modelo_cv, X_cv, y_cv, cv=kf10, scoring="neg_mean_squared_error")

# LOO
loo = LeaveOneOut()
scores_loo = cross_val_score(modelo_cv, X_cv, y_cv, cv=loo, scoring="neg_mean_squared_error")

resultados = {
    "Hold-Out\n(30 reps)": np.array(holdout_scores),
    "5-Fold CV": scores_kf5,
    "10-Fold CV": scores_kf10,
    f"LOO (N={len(X_cv)})": scores_loo,
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
nomes = list(resultados.keys())
medias = [-v.mean() for v in resultados.values()]
stds   = [v.std()   for v in resultados.values()]
cores_cv = ["steelblue", "green", "darkorange", "purple"]
bars = ax.bar(nomes, medias, color=cores_cv, alpha=0.75,
              yerr=stds, capsize=5, edgecolor="black")
ax.set_title("MSE médio por método de CV")
ax.set_ylabel("MSE (menor = melhor)")
ax.grid(alpha=0.3, axis="y")

ax = axes[1]
ax.boxplot([np.abs(v) for v in resultados.values()],
           labels=nomes, patch_artist=True,
           boxprops=dict(facecolor="lightblue"))
ax.set_title("Distribuição dos scores por fold")
ax.set_ylabel("|MSE| por fold")
ax.grid(alpha=0.3, axis="y")

plt.suptitle("Comparação de variantes de Cross-Validation", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("Resumo:")
for nome, scores in resultados.items():
    print(f"  {nome.replace(chr(10), ' '):20s}  MSE = {-scores.mean():.1f} ± {scores.std():.1f}")

In [ ]:
# === Visualizar padrão de splits por variante ===

N_vis = 20   # número de amostras para visualização
X_vis = np.arange(N_vis).reshape(-1, 1)
y_vis = np.zeros(N_vis)

fig, axes = plt.subplots(4, 1, figsize=(12, 8))

# Hold-Out (1 split fixo 75/25)
ax = axes[0]
tr_idx, vl_idx = train_test_split(np.arange(N_vis), test_size=0.25, random_state=0)
mapa = np.zeros(N_vis)
mapa[vl_idx] = 1
ax.pcolor([mapa], cmap="RdYlGn", vmin=0, vmax=1, edgecolors="white", linewidth=1)
ax.set_yticks([0.5]); ax.set_yticklabels(["Hold-Out"])
ax.set_xlim(0, N_vis); ax.set_xticks([])

# K-Fold K=5
ax = axes[1]
splits_kf = list(KFold(n_splits=5, shuffle=False).split(X_vis))
mapa_kf = np.zeros((5, N_vis))
for i, (tr, vl) in enumerate(splits_kf):
    mapa_kf[i, vl] = 1
ax.pcolor(mapa_kf, cmap="RdYlGn", vmin=0, vmax=1, edgecolors="white", linewidth=1)
ax.set_yticks(np.arange(5)+0.5); ax.set_yticklabels([f"Fold {i+1}" for i in range(5)])
ax.set_xlabel(""); ax.set_title("K-Fold (K=5)")
ax.set_xlim(0, N_vis); ax.set_xticks([])

# Stratified K-Fold
ax = axes[2]
y_cls = (np.arange(N_vis) % 3).astype(int)  # 3 classes simuladas
splits_skf = list(StratifiedKFold(n_splits=5, shuffle=False).split(X_vis, y_cls))
mapa_skf = np.zeros((5, N_vis))
for i, (tr, vl) in enumerate(splits_skf):
    mapa_skf[i, vl] = 1
ax.pcolor(mapa_skf, cmap="RdYlGn", vmin=0, vmax=1, edgecolors="white", linewidth=1)
ax.set_yticks(np.arange(5)+0.5); ax.set_yticklabels([f"Fold {i+1}" for i in range(5)])
ax.set_title("Stratified K-Fold (K=5)")
ax.set_xlim(0, N_vis); ax.set_xticks([])

# LOO
ax = axes[3]
mapa_loo = np.zeros((N_vis, N_vis))
for i in range(N_vis):
    mapa_loo[i, i] = 1
ax.pcolor(mapa_loo, cmap="RdYlGn", vmin=0, vmax=1, edgecolors="white", linewidth=0.5)
ax.set_yticks([]); ax.set_title(f"Leave-One-Out (N={N_vis} treinos)")
ax.set_xlim(0, N_vis)

p1 = mpatches.Patch(color="green", label="Treino")
p2 = mpatches.Patch(color="red",   label="Validação")
fig.legend(handles=[p1, p2], loc="upper right", fontsize=10)
plt.suptitle("Padrão de splits — verde = treino, vermelho = validação", fontsize=12)
plt.tight_layout()
plt.show()

---

## Seção 7 — Bootstrapping

### 7.1 Conceito Teórico

**Bootstrapping** é uma técnica de **resampling** para quantificar a incerteza de um estimador sem precisar de fórmulas analíticas fechadas.

**Ideia central:**
- Muitos estimadores têm distribuições amostrais complexas (sem forma conhecida)
- A **aproximação Bootstrap** trata o dataset observado $D$ como uma **população empírica**
- Reamostrar $D$ **com reposição** simula o processo de coletar novos datasets da população real
- A variabilidade entre as reamostras aproxima a incerteza do estimador

**Usos no statistical learning:**
- **Coefficient uncertainty:** quão estáveis são os parâmetros estimados?
- **Feature importance:** com que frequência uma variável permanece importante?
- **Prediction uncertainty:** distribuição das previsões sobre modelos bootstrap
- **Model evaluation:** distribuição bootstrap de AUC, RMSE, MAE, accuracy
- **Bagging:** média de modelos treinados em amostras bootstrap → reduz variância

### 7.2 Algoritmo — Nonparametric Bootstrap

```
Input: dados D = {z₁, …, zₙ}, estatística S, replicatas B

for b = 1, …, B:
    Amostra Dᵇ: n observações de D com reposição
    Calcula sᵇ = S(Dᵇ)
end

Retorna distribuição {s¹, …, sᴮ}

Saídas comuns:
    erro padrão = std({sᵇ})
    IC percentil = quantis de {sᵇ}
```

> **Importante:** use o mesmo pipeline completo dentro de S — preprocessing, feature selection, tuning — para evitar vazamento de informação.

### 7.3 Bootstrap Confidence Intervals

| Tipo | Fórmula | Quando usar |
|------|---------|-------------|
| **Normal approximation** | $\hat{s} \pm 1.96 \cdot \text{std}(\{s^b\})$ | Distribuição bootstrap aproximadamente Normal |
| **Percentile interval** | $[q_{2.5\%}(\{s^b\}),\; q_{97.5\%}(\{s^b\})]$ | Distribuição assimétrica; mais robusto |
| **BCa (Bias-Corrected Accelerated)** | Corrige viés e aceleração da distribuição | Distribuição enviesada ou assimétrica; mais preciso mas mais complexo |

In [ ]:
# === Hands-on: Nonparametric Bootstrap — Incerteza nos Coeficientes ===

np.random.seed(42)
N_boot = 80
p_boot = 5

X_boot, y_boot = make_regression(n_samples=N_boot, n_features=p_boot,
                                   n_informative=3, noise=20, random_state=42)
X_boot = StandardScaler().fit_transform(X_boot)

B = 500   # número de replicatas bootstrap
coefs_boot = np.zeros((B, p_boot))
rmse_boot  = np.zeros(B)

modelo_boot = Ridge(alpha=1.0)

for b in range(B):
    idx = np.random.choice(N_boot, size=N_boot, replace=True)
    oob = np.setdiff1d(np.arange(N_boot), idx)   # out-of-bag
    modelo_boot.fit(X_boot[idx], y_boot[idx])
    coefs_boot[b] = modelo_boot.coef_
    if len(oob) > 0:
        preds_oob = modelo_boot.predict(X_boot[oob])
        rmse_boot[b] = np.sqrt(np.mean((y_boot[oob] - preds_oob)**2))

# Coeficiente original (no dataset completo)
modelo_boot.fit(X_boot, y_boot)
coefs_orig = modelo_boot.coef_

fig, axes = plt.subplots(1, p_boot, figsize=(15, 5), sharey=False)
for j, ax in enumerate(axes):
    ax.hist(coefs_boot[:, j], bins=30, color="steelblue", alpha=0.7, density=True)
    ci_lo = np.percentile(coefs_boot[:, j], 2.5)
    ci_hi = np.percentile(coefs_boot[:, j], 97.5)
    ax.axvline(coefs_orig[j], color="red",    lw=2, label=f"Orig={coefs_orig[j]:.2f}")
    ax.axvline(ci_lo, color="orange", ls="--", lw=1.5, label=f"IC 95%")
    ax.axvline(ci_hi, color="orange", ls="--", lw=1.5)
    ax.axvline(0, color="black", lw=0.8, ls=":")
    ax.set_title(f"Feature {j+1}", fontsize=10)
    ax.set_xlabel(f"θ_{j+1}")
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

plt.suptitle("Bootstrap Coefficient Uncertainty (B=500 replicatas)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print("Coeficientes originais vs IC Bootstrap 95% (Percentile):")
for j in range(p_boot):
    ci_lo = np.percentile(coefs_boot[:, j], 2.5)
    ci_hi = np.percentile(coefs_boot[:, j], 97.5)
    sig = "*" if ci_lo > 0 or ci_hi < 0 else " "
    print(f"  Feature {j+1}: {coefs_orig[j]:+.3f}  IC=[{ci_lo:.3f}, {ci_hi:.3f}] {sig}")
print("  * = IC não inclui zero (feature provavelmente significativa)")

In [ ]:
# === Comparar tipos de Bootstrap Confidence Intervals ===

estatistica_boot = coefs_boot[:, 0]   # usar coeficiente da Feature 1
s_orig = coefs_orig[0]

# Normal approximation
se = estadistica_boot.std()
ci_normal = (s_orig - 1.96*se, s_orig + 1.96*se)

# Percentile interval
ci_perc = (np.percentile(estatistica_boot, 2.5), np.percentile(estatistica_boot, 97.5))

# BCa — bias-corrected and accelerated (implementação simplificada)
z0 = stats.norm.ppf(np.mean(estatistica_boot < s_orig))  # bias correction
jackknife_vals = np.array([
    Ridge(alpha=1.0).fit(
        np.delete(X_boot, i, axis=0), np.delete(y_boot, i)
    ).coef_[0]
    for i in range(min(N_boot, 30))  # LOO jack (limitado a 30 para velocidade)
])
jk_mean = jackknife_vals.mean()
num = np.sum((jk_mean - jackknife_vals)**3)
den = 6 * (np.sum((jk_mean - jackknife_vals)**2))**1.5
a = num / den if den != 0 else 0.0   # acceleration

alpha_lo = stats.norm.cdf((z0 + (z0 + stats.norm.ppf(0.025)) / (1 - a*(z0 + stats.norm.ppf(0.025)))))
alpha_hi = stats.norm.cdf((z0 + (z0 + stats.norm.ppf(0.975)) / (1 - a*(z0 + stats.norm.ppf(0.975)))))
ci_bca = (np.percentile(estatistica_boot, 100*alpha_lo),
          np.percentile(estatistica_boot, 100*alpha_hi))

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(estatistica_boot, bins=40, color="steelblue", alpha=0.6, density=True, label="Bootstrap dist.")

for (lo, hi), (nome, cor) in zip(
        [ci_normal, ci_perc, ci_bca],
        [("Normal approx.", "red"), ("Percentile", "green"), ("BCa", "purple")]):
    ax.axvline(lo, color=cor, lw=2, ls="--")
    ax.axvline(hi, color=cor, lw=2, ls="--", label=f"{nome}: [{lo:.2f}, {hi:.2f}]")

ax.axvline(s_orig, color="black", lw=2, label=f"Estimativa original: {s_orig:.3f}")
ax.set_title("Três tipos de Bootstrap Confidence Intervals (95%) — Feature 1")
ax.set_xlabel("θ₁")
ax.set_ylabel("Densidade")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Bootstrap Confidence Intervals (95%) para θ₁:")
print(f"  Normal approx.: [{ci_normal[0]:.3f}, {ci_normal[1]:.3f}]")
print(f"  Percentile:     [{ci_perc[0]:.3f}, {ci_perc[1]:.3f}]")
print(f"  BCa:            [{ci_bca[0]:.3f}, {ci_bca[1]:.3f}]")

In [ ]:
# === Bagging: média de modelos bootstrap reduz variância ===

np.random.seed(10)
N_bag = 60

x_bag = np.sort(np.random.uniform(-3, 3, N_bag))
y_bag = np.sin(x_bag) + np.random.normal(0, 0.4, N_bag)

from numpy.polynomial import polynomial as P

def poly_fit_predict(x_tr, y_tr, x_pred, grau=8):
    coefs = np.polyfit(x_tr, y_tr, grau)
    return np.polyval(coefs, x_pred)

x_pred = np.linspace(-3, 3, 300)
B_bag = 200
preds_individuais = np.zeros((B_bag, len(x_pred)))

for b in range(B_bag):
    idx = np.random.choice(N_bag, size=N_bag, replace=True)
    try:
        preds_individuais[b] = poly_fit_predict(x_bag[idx], y_bag[idx], x_pred, grau=7)
    except:
        preds_individuais[b] = np.zeros(len(x_pred))

pred_bagging  = preds_individuais.mean(axis=0)
pred_simples  = poly_fit_predict(x_bag, y_bag, x_pred, grau=7)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
for b in range(0, B_bag, 20):
    ax.plot(x_pred, preds_individuais[b], alpha=0.15, color="steelblue", lw=1)
ax.plot(x_pred, pred_simples, "r-", lw=2, label="Modelo único (high variance)")
ax.scatter(x_bag, y_bag, s=20, color="gray", zorder=3, alpha=0.6)
ax.plot(x_pred, np.sin(x_pred), "k--", lw=2, label="Verdadeiro sin(x)")
ax.set_title("Modelos individuais (bootstrap) — alta variância")
ax.set_ylim(-3, 3); ax.legend(fontsize=9); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(x_pred, pred_bagging, "g-", lw=2.5, label=f"Bagging (B={B_bag}) — baixa variância")
ax.plot(x_pred, np.sin(x_pred), "k--", lw=2, label="Verdadeiro sin(x)")
ax.fill_between(x_pred,
                preds_individuais.mean(0) - preds_individuais.std(0),
                preds_individuais.mean(0) + preds_individuais.std(0),
                alpha=0.2, color="green", label="± 1 std bootstrap")
ax.scatter(x_bag, y_bag, s=20, color="gray", zorder=3, alpha=0.6)
ax.set_title("Bagging: média das B previsões bootstrap")
ax.set_ylim(-3, 3); ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.suptitle("Bagging = Bootstrap + Averaging → reduz variância", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# Erro empírico
mse_simples  = np.mean((np.sin(x_pred) - pred_simples)**2)
mse_bagging  = np.mean((np.sin(x_pred) - pred_bagging)**2)
print(f"MSE (modelo único): {mse_simples:.4f}")
print(f"MSE (bagging):      {mse_bagging:.4f}")
print(f"Redução relativa:   {(mse_simples - mse_bagging)/mse_simples*100:.1f}%")

---

## Resumo — Mapa Mental da Aula

```
PROBABILISTIC VIEW
  ├── Likelihood p(D|θ)  →  -log = MSE / SSE
  ├── Prior p(θ)
  │     ├── Gaussiano  →  -log = ||θ||²   →  L2 / Ridge
  │     └── Laplace    →  -log = ||θ||₁   →  L1 / Lasso
  └── Posterior p(θ|D) ∝ Likelihood × Prior
        └── MAP = minimizar MSE + λ·Regularização

DECISION PROCESS
  ├── Small data, many features → L2/L1, Early Stopping
  ├── Structured outputs        → TV/Laplace (Output Reg.)
  ├── Strong prior knowledge    → Regularization by Design
  └── Unstable optimization     → Gradient Clipping, Max-Norm

HYPERPARAMETER CHOICE
  ├── Guidelines: L²=1e-6…1e-2, L¹=1e-6…1e-3, Dropout=0.1…0.5
  ├── Search: log-grid → Coarse-to-Fine
  ├── Average 3–5 seeds se N pequeno
  └── Retreinar em Train+Val; reportar no Test UMA VEZ

DIAGNOSIS
  ├── Under-regularização: gap Train/Val GRANDE → aumentar λ
  └── Over-regularização:  ambos ALTOS          → reduzir λ

CROSS-VALIDATION
  ├── Hold-Out  → rápido, N grande
  ├── K-Fold    → padrão (K=5 ou 10)
  ├── Stratified → classificação desbalanceada
  └── LOO       → N muito pequeno, custoso

BOOTSTRAPPING
  ├── Resampling with replacement
  ├── Confidence Intervals: Normal, Percentile, BCa
  └── Bagging: média bootstrap → reduz variância
```

---

### Exercícios para fixar

1. **Probabilistic View:** implemente a solução fechada do Ridge $\hat{\theta} = (X^TX + \lambda I)^{-1}X^Ty$ e compare com `ajustar_map`. Os resultados coincidem?
2. **Prior:** mude o prior de Gaussiano para uniforme (sem regularização). Como isso afeta o MAP?
3. **Decision Process:** no cenário de Structured Outputs, pesquise Total Variation (TV) e implemente-a manualmente para um sinal 1D.
4. **Coarse-to-Fine:** aplique a busca em `Lasso` em vez de `Ridge` e compare os λ* encontrados.
5. **Cross-Validation:** use `StratifiedKFold` com um dataset de classificação desbalanceado e compare com `KFold` padrão.
6. **Bootstrap:** implemente o BCa completo usando a fórmula do slide e compare com o IC percentile. Quando diferem mais?

---

### Referências neste repositório

- `L1_General_Linear_Models_Basis_Functions.ipynb` — Design Matrix, Basis Functions
- `L2_Bias_Variance_Model_Selection.ipynb` — Bias-Variance Trade-Off, Model Selection Workflow
- `L3_Regularization_CrossValidation.ipynb` — Hard/Soft Regularization, Early Stopping, Dropout
- `lectures/Statistisches_Lernen_2_v4.pdf` — material completo desta aula